# 02 正向退出（1→0）：公司端独立大候选池

本 Notebook 只重建 `plus` 侧的大候选池。公司端会重新计算每个候选的 Development 阈值，在该侧池内用 Development+Validation 冻结一个 Top1，冻结后才展示 Test。另一侧不在本 Notebook 中运行。

In [ ]:
# 02 正向退出（1→0）：公司端独立大候选池
# This side is fixed in this notebook; there is no manual side switch.
from pathlib import Path
import sys
import pandas as pd
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 240)
pd.set_option('display.max_colwidth', 48)

PACKAGE_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src" / "pool_runner.py").is_file()
)
sys.path.insert(0, str(PACKAGE_ROOT / "src"))
from pool_runner import run_company_side

result = run_company_side("plus")
assert result["test_used_for_selection"] is False
print("status:", result["status"], "side:", result["side"], "pool_rule:", result["pool_rule_id"])
print("提示：上面的 [1545] 行是运行进度；下面各格是可直接截图的审计、冻结和全周期报告。")


## 输入、1545 对照和实际交易日对齐

In [ ]:
audit = result['input_audit']
frozen = audit['frozen_1545']
print('audit_passed:', audit['audit_passed'])
print('1545 states/counts:', frozen['unique_values'], frozen['counts'])
print('reference parity:', frozen['reference_parity'])
print('alignment:', audit['alignment'])
print('forbidden feature hits:', audit['forbidden_feature_hits'])
assert audit['audit_passed'] and audit['forbidden_feature_hits'] == []


## 冻结前：全量合成候选池与 Development+Validation 排名

In [ ]:
pool_summary = pd.DataFrame([{
    'side': result['side'],
    'versions': ' + '.join(result['pool']['versions']),
    'raw_candidates': result['pool']['raw_candidate_count'],
    'metric_rows_before_cross_version_dedup': result['pool']['metric_rows_before_cross_version_dedup'],
    'unique_signal_rows': result['pool']['unique_signal_rows'],
    'ranked_rows': result['pool']['ranked_rows'],
}])
print('候选池摘要：')
display(pool_summary)
ranking_cols = [
    'rank', 'version', 'core_logic_name', 'candidate_id',
    'development_n', 'development_h1_improvement_bp',
    'validation_n', 'validation_h1_improvement_bp',
    'pooled_n', 'pooled_h1_improvement_bp', 'joint_score', 'formal_pass',
]
print('冻结前 DV 排名前 20：')
display(result['top20'][[c for c in ranking_cols if c in result['top20'].columns]].head(20).round(4))
print('版内各逻辑 DV-Top1 与冻结后 Test（仅观察，不参与最终大池排序）：')
version_cols = [
    'version', 'core_logic_name', 'candidate_id',
    'full_cycle_n', 'full_cycle_h1_improvement_bp', 'full_cycle_h1_median_bp',
    'development_n', 'development_h1_improvement_bp',
    'validation_n', 'validation_h1_improvement_bp',
    'test_n', 'test_h1_improvement_bp', 'test_h3_improvement_bp',
    'test_used_for_selection',
]
version_table = pd.DataFrame(result.get('version_top1_test', []))
display(version_table[[c for c in version_cols if c in version_table.columns]].round(4))


## 最终报告：冻结参数、全周期、分周期与 Test 年度结果

In [ ]:
print('=' * 88)
print('最终冻结参数（Top1 已由 Development+Validation 冻结；Test 仅在此后观察）')
freeze_cols = [
    'rank', 'version', 'side', 'action', 'core_logic_name', 'logic_title_cn',
    'score_variant', 'threshold_quantile', 'threshold_value', 'min_state_age',
    'confirm_days', 'cooldown_days', 'model_fit_period',
    'threshold_fit_period', 'ranking_period', 'selection_data_end',
    'test_used_for_selection',
]
freeze_table = pd.DataFrame({
    'freeze_parameter': [c for c in freeze_cols if c in result['freeze']],
    'value': [result['freeze'][c] for c in freeze_cols if c in result['freeze']],
})
display(freeze_table)
print('冻结信号表现：全周期 + Development + Validation + Test')
all_periods = pd.DataFrame([result['full_cycle'], *result['periods']])
core_metric_cols = ['period', 'n', 'eligible_n', 'coverage', 'h1_improvement_bp', 'h1_median_bp', 'h1_win_rate']
display(all_periods[core_metric_cols].round(4))
aux_metric_cols = ['period', 'h3_improvement_bp', 'c2c_observation_bp', 'test_used_for_selection']
display(all_periods[aux_metric_cols].round(4))
print('Test 年度拆分（冻结后观察）：')
display(pd.DataFrame(result['year_metrics']).round(4))
print('冻结后诊断（不参与筛选）：')
display(pd.DataFrame({k: v['bootstrap_o2o_h1'] for k, v in result['diagnostics'].items()}).T.round(4))
print('最近 10 条冻结信号：')
recent_cols = ['formation_date', 'side', 'effective_date', 'exit_h1_date', 'state_age', 'o2o_h1', 'o2o_h3', 'c2c_obs']
display(result['recent_signals'].tail(10)[recent_cols].round(6))
assert result['freeze']['test_used_for_selection'] is False
print('FINAL_REPORT_END | Test_used_for_selection = False')
